In [ ]:
!pip install fastapi uvicorn pydantic transformers torch nest-asyncio pyngrok accelerated-scan


In [ ]:
import nest_asyncio
import torch
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
from huggingface_hub import login
login()

In [ ]:

NGROK_AUTH_TOKEN = "3FGklSY3sPzXlNIqs73XaNPX2PD_4fKB7KVZ7tf6VR46fmcAr"

MODEL_NAME = "rose00009/Code_Review_Assistant_Model1"

# INITIALIZE FASTAPI
app = FastAPI(
    title="Qwen2.5-Coder Cloud Inference Gateway",
    description="High-performance GPU endpoint for code reviews."
)

class CodePayload(BaseModel):
    code: str

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
print("Model successfully mapped and ready for cloud inference")

@app.post("/review")
def process_code_review(payload: CodePayload):
    structured_prompt = (
        "<|im_start|>system\n"
        "You are an expert code reviewer. Analyze the following source code for "
        "performance bottlenecks, structural optimization, and logical security vulnerabilities. "
        "Provide clear, actionable code critique.<|im_end|>\n"
        f"<|im_start|>user\n{payload.code}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

    inputs = tokenizer(structured_prompt, return_tensors="pt").to("cuda")
    input_length = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.4,
            top_p=0.9,
            do_sample=True
        )

    #Slice out the prompt tokens using input_length
    feedback_only = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
    return {"review": feedback_only}

# Configure and establish the public web bridge proxy connection
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_tunnel = ngrok.connect(8000)

print("\n" + "="*60)
print("CLOUD BACKEND ENGINE IS RUNNING")
print(f"   {public_tunnel.public_url}/review")
print("="*60 + "\n")

# Apply patch to let Uvicorn run loop operations natively inside Colab notebooks
nest_asyncio.apply()
config = uvicorn.Config(app, host="127.0.0.1", port=8000, loop="asyncio")
server = uvicorn.Server(config)

await server.serve()

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model successfully mapped and ready for cloud inference

CLOUD BACKEND ENGINE IS RUNNING
   https://petticoat-existing-staleness.ngrok-free.dev/review



INFO:     Started server process [835]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     92.99.209.186:0 - "POST /review HTTP/1.1" 200 OK
INFO:     92.99.209.186:0 - "POST /review HTTP/1.1" 200 OK
INFO:     92.99.209.186:0 - "POST /review HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [835]
